In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, max, min, sum, round, to_date
from pyspark.sql.window import Window

In [0]:
# Load the Silver table
silver_df = spark.table("jarvis_server_1.silver.stock_data")  # replace with your Silver table name

# Quick check
silver_df.show(5)

+------+----------+------+-------+------+------+--------+----------------+
| close|      date|  high|    low|  open|symbol|  volume|daily_return_pct|
+------+----------+------+-------+------+------+--------+----------------+
|272.14|2026-02-24|274.89| 267.71|267.86|  AAPL|47014619|             1.6|
|266.18|2026-02-23|269.43|263.381|263.49|  AAPL|37308155|            1.02|
|264.58|2026-02-20|264.75| 258.16|258.97|  AAPL|42070499|            2.17|
|260.58|2026-02-19|264.48| 260.05| 262.6|  AAPL|30845294|           -0.77|
|264.35|2026-02-18|266.82| 262.45| 263.6|  AAPL|34203337|            0.28|
+------+----------+------+-------+------+------+--------+----------------+
only showing top 5 rows


In [0]:
# Add 7-day and 30-day moving average for close price
window_7 = Window.partitionBy("symbol").orderBy("date").rowsBetween(-6, 0)
window_30 = Window.partitionBy("symbol").orderBy("date").rowsBetween(-29, 0)

gold_df = silver_df.withColumn("ma_7_close", round(avg(col("close")).over(window_7),2)) \
                   .withColumn("ma_30_close", round(avg(col("close")).over(window_30),2)) \
                   .withColumn("daily_return_pct", round(col("daily_return_pct"), 2))

In [0]:
from pyspark.sql.functions import month, year

monthly_df = silver_df.withColumn("year", year("date")) \
                      .withColumn("month", month("date")) \
                      .groupBy("symbol","year","month") \
                      .agg(
                          round(avg("close"),2).alias("avg_close"),
                          round(max("high"),2).alias("max_high"),
                          round(min("low"),2).alias("min_low"),
                          round(avg("daily_return_pct"),2).alias("avg_daily_return_pct"),
                          sum("volume").alias("total_volume")
                      )

In [0]:
# Save Gold table
gold_df.write.mode("overwrite").saveAsTable("gold_stock_data")
monthly_df.write.mode("overwrite").saveAsTable("gold_stock_monthly")

In [0]:
# Preview the first 5 rows of the daily Gold table
gold_df.show(5)

# Preview the first 5 rows of the monthly Gold table
monthly_df.show(5)

# Or use display() in Databricks for a nicer table view
display(gold_df)
display(monthly_df)

+------+----------+------+------+-------+------+--------+----------------+----------+-----------+
| close|      date|  high|   low|   open|symbol|  volume|daily_return_pct|ma_7_close|ma_30_close|
+------+----------+------+------+-------+------+--------+----------------+----------+-----------+
|255.45|2025-10-01|258.79|254.93| 255.04|  AAPL|48713940|            0.16|    255.45|     255.45|
|257.13|2025-10-02|258.18|254.15|256.575|  AAPL|42630239|            0.22|    256.29|     256.29|
|258.02|2025-10-03|259.24|253.95|254.665|  AAPL|49155614|            1.32|    256.87|     256.87|
|256.69|2025-10-06|259.07|255.05| 257.99|  AAPL|44664118|            -0.5|    256.82|     256.82|
|256.48|2025-10-07| 257.4|255.43|256.805|  AAPL|31955776|           -0.13|    256.75|     256.75|
+------+----------+------+------+-------+------+--------+----------------+----------+-----------+
only showing top 5 rows
+------+----+-----+---------+--------+-------+--------------------+------------+
|symbol|year|

close,date,high,low,open,symbol,volume,daily_return_pct,ma_7_close,ma_30_close
255.45,2025-10-01,258.79,254.93,255.04,AAPL,48713940,0.16,255.45,255.45
257.13,2025-10-02,258.18,254.15,256.575,AAPL,42630239,0.22,256.29,256.29
258.02,2025-10-03,259.24,253.95,254.665,AAPL,49155614,1.32,256.87,256.87
256.69,2025-10-06,259.07,255.05,257.99,AAPL,44664118,-0.5,256.82,256.82
256.48,2025-10-07,257.4,255.43,256.805,AAPL,31955776,-0.13,256.75,256.75
258.06,2025-10-08,258.52,256.11,256.52,AAPL,36496895,0.6,256.97,256.97
254.04,2025-10-09,258.0,253.14,257.805,AAPL,38322012,-1.46,256.55,256.55
245.27,2025-10-10,256.38,244.0,254.94,AAPL,61999098,-3.79,255.1,255.14
247.66,2025-10-13,249.69,245.56,249.38,AAPL,38142942,-0.69,253.75,254.31
247.77,2025-10-14,248.845,244.7,246.6,AAPL,35477986,0.47,252.28,253.66


symbol,year,month,avg_close,max_high,min_low,avg_daily_return_pct,total_volume
AAPL,2026,2,268.94,280.91,255.45,0.18,849899855
AAPL,2026,1,257.65,277.84,243.42,-0.22,1036170325
AAPL,2025,12,276.31,288.62,266.95,-0.11,922283649
AAPL,2025,11,271.66,280.38,265.32,0.31,876481453
AAPL,2025,10,258.3,277.32,244.0,-0.07,1097142365
MSFT,2026,2,402.73,430.74,381.71,-0.74,669511012
MSFT,2026,1,465.05,489.7,421.02,-0.12,683781215
MSFT,2025,12,483.86,493.5,470.88,0.19,494657629
MSFT,2025,11,496.8,524.96,464.89,-0.2,479340345
MSFT,2025,10,521.2,553.72,506.0,-0.17,468447068
